In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#pip install pandas numpy matplotlib seaborn openpyxl statsmodels plotly

# 1.Load dataset

In [ ]:
# Load the Accessories sheet from the Excel file
accessories_data = pd.read_excel('/content/EIU case study - Sample data - Accessories.xlsx')

# Display the first few rows to inspect the data
accessories_data.head()

,Customer Name,Reporting Business Type,Collective Class,Series No,Item SKU,Item Description,Colors,Series Color,Item Grouping,AFI Sales Category,Import/Domestic Code,Market Introduced At,Parent Style Description,Warehouse,Order Quantity,FiscalMonthLastDate,Status
0,Customer-1,Others,ACCESSORIES,A2000550,A2000550,VASE (3/CS),Black,Black,Table Top,Sig. Ashley Accessories,I,QTR 1 2022,Casual,1,1,2023-05-27,Invoiced
1,Customer-2,eCommerce,ACCESSORIES,A4000107,A4000107,CONSOLE SOFA TABLE,Antique White/Brown,Antique White/Brown,Accent Furniture,Sig. Ashley Accessories,I,Las Vegas January 2018,Casual,17,1,2024-08-24,Invoiced
2,Customer-3,Others,ACCESSORIES,A3000301,A3000301,UPHOLSTERED ACCENT BENCH,Charcoal/Brown,Charcoal/Brown,Accent Seating,Sig. Ashley Accessories,I,High Point October 2020,Casual,1,1,2023-01-28,Invoiced
3,Customer-4,Others,ACCESSORIES,A3000271,A3000271,ACCENT CHAIR,Ivory/Brown,Ivory/Brown,Accent Seating,Sig. Ashley Accessories,I,High Point October 2020,Casual,5,3,2023-04-01,Invoiced
4,Customer-5,Others,ACCESSORIES,A2C00203,A2C00203,ACCESSORY SET (5/CN),Multi,Multi,Table Top,Sig. Ashley Accessories,I,QTR 2 2022,Casual,15,1,2023-04-29,Invoiced


# 2.Convert to wide format & Cleaning

In [ ]:
import pandas as pd

# -------------------------------
# Step 0: Data Preparation and Cleaning
# -------------------------------
# Giả sử accessories_data đã được load, ví dụ:
# accessories_data = pd.read_excel("YourFile.xlsx", sheet_name="Outdoor")
df = accessories_data[[
    "FiscalMonthLastDate", "Warehouse", "Series No", "Item SKU", "Order Quantity"
]].copy().dropna()

# Đổi tên cột: thay khoảng trắng bằng dấu "-" để dễ xử lý
df.columns = [col.replace(" ", "-") for col in df.columns]

# Chuyển FiscalMonthLastDate sang kiểu datetime
df["FiscalMonthLastDate"] = pd.to_datetime(df["FiscalMonthLastDate"])

# -------------------------------
# Step 1: Aggregate Data at the Series Level
# -------------------------------
# Chúng ta chỉ quan tâm đến cột "Series No". Mỗi hàng sẽ được gộp theo FiscalMonthLastDate và Series-No.
df_agg = df.groupby(["FiscalMonthLastDate", "Series-No"])["Order-Quantity"].sum().reset_index()

# -------------------------------
# Step 2: Convert to Wide Format for Forecasting
# -------------------------------
# Pivot table: index = FiscalMonthLastDate, các cột là các giá trị của "Series-No"
df_wide = df_agg.pivot_table(
    index="FiscalMonthLastDate",
    columns="Series-No",
    values="Order-Quantity",
    fill_value=0
)

# -------------------------------
# Step 3: Compute Hierarchical Aggregations (2-level: Total and Series)
# -------------------------------
# 3.1: Total (Overall Sum): tổng của tất cả các Series
df_total = df_wide.sum(axis=1).to_frame(name=("Total",))

# 3.2: Series-level: giữ lại dữ liệu của các Series, đặt tên cột thành MultiIndex (1 cấp)
df_series = df_wide.copy()
df_series.columns = pd.MultiIndex.from_tuples([(col,) for col in df_series.columns])

# -------------------------------
# Step 4: Merge Total and Series-level Data
# -------------------------------
df_hierarchical = pd.concat([df_total, df_series], axis=1)

# -------------------------------
# Step 5: Sort Columns Hierarchically
# -------------------------------
# Sắp xếp sao cho cột "Total" đứng đầu, sau đó các Series được sắp xếp theo tên (alphabet)
ordered_columns = sorted(
    df_hierarchical.columns,
    key=lambda x: (0 if x[0] == "Total" else 1, x[0])
)
df_hierarchical = df_hierarchical[ordered_columns]

# -------------------------------
# Step 6: Flatten MultiIndex Columns
# -------------------------------
# Vì các cột Series đã có MultiIndex 1 cấp, ta chỉ lấy phần tử đầu tiên của mỗi tuple
df_hierarchical.columns = [col[0] for col in df_hierarchical.columns]

# -------------------------------
# (Optional) Replace specific dates in the index if needed
# -------------------------------
df_hierarchical.index = df_hierarchical.index.to_series().replace({
    pd.Timestamp("2023-04-01"): pd.Timestamp("2023-03-25"),
    pd.Timestamp("2023-07-01"): pd.Timestamp("2023-06-24")
})

df_hierarchical

,Total,36206,36207,36208,36229,36230,36232,53302,53303,A1000954,...,T505-106,T505-560,T505-562,T505-662,T505-742,T505-762,T505-962,T800,T800-111,T800-124
FiscalMonthLastDate,,,,,,,,,,,,,,,,,,,,,
2022-01-22,94220.0,272.0,370.0,442.0,120.0,145.0,135.0,18.0,158.0,15.0,...,92.0,243.0,109.0,122.0,99.0,286.0,80.0,184.0,339.0,193.0
2022-02-19,105423.0,272.0,479.0,590.0,122.0,173.0,282.0,37.0,213.0,6.0,...,119.0,227.0,97.0,125.0,128.0,262.0,120.0,183.0,327.0,179.0
2022-03-26,131285.0,277.0,562.0,573.0,267.0,274.0,343.0,15.0,226.0,19.0,...,109.0,234.0,151.0,87.0,117.0,314.0,115.0,275.0,369.0,740.0
2022-04-23,72222.0,165.0,198.0,320.0,104.0,98.0,143.0,27.0,120.0,1.0,...,117.0,160.0,90.0,149.0,77.0,223.0,83.0,171.0,312.0,326.0
2022-05-21,75760.0,157.0,201.0,231.0,162.0,144.0,144.0,16.0,140.0,2.0,...,76.0,144.0,76.0,194.0,79.0,164.0,81.0,155.0,253.0,127.0
2022-06-25,85249.0,153.0,213.0,351.0,181.0,168.0,200.0,11.0,144.0,4.0,...,96.0,177.0,84.0,69.0,68.0,342.0,71.0,250.0,454.0,473.0
2022-07-23,71975.0,111.0,175.0,201.0,115.0,177.0,179.0,20.0,157.0,6.0,...,78.0,164.0,50.0,58.0,90.0,488.0,43.0,144.0,87.0,77.0
2022-08-20,77834.0,120.0,255.0,370.0,109.0,70.0,169.0,3.0,165.0,1.0,...,71.0,275.0,99.0,228.0,95.0,335.0,113.0,116.0,112.0,52.0
2022-09-24,107000.0,232.0,222.0,368.0,166.0,192.0,246.0,6.0,283.0,1.0,...,834.0,277.0,151.0,134.0,178.0,707.0,141.0,197.0,452.0,100.0


# 3.Classify the Series into groups a, b, c and x, y, z according to sales

In [ ]:
# -------------------------------
# Phần 2: Phân loại các Series thành nhóm a, b, c theo doanh số
# -------------------------------
series_cols = [col for col in df_hierarchical.columns if col != "Total"]

# Tính tổng doanh số cho mỗi Series
series_totals = df_hierarchical[series_cols].sum(axis=0).reset_index()
series_totals.columns = ['Series-No', 'Total_Sales']
overall_total = series_totals['Total_Sales'].sum()
n_total = len(series_totals)

# Sắp xếp giảm dần theo Total_Sales
series_totals = series_totals.sort_values(by='Total_Sales', ascending=False).reset_index(drop=True)

# Nhóm a: chọn các series theo thứ tự giảm dần cho đến khi tích lũy ≥80% overall sales,
#         nhưng số lượng không vượt quá 20% tổng số series.
max_count_a = int(0.2 * n_total)
cumulative_sales_a = 0
group_a = []
for idx, row in series_totals.iterrows():
    group_a.append(row['Series-No'])
    cumulative_sales_a += row['Total_Sales']
    if (cumulative_sales_a / overall_total >= 0.80) or (len(group_a) >= max_count_a):
        break

# Nhóm b: từ các series còn lại, chọn đến khi tích lũy ≥15% overall sales,
#         nhưng số lượng không vượt quá 30% số series còn lại.
remaining_df = series_totals[~series_totals['Series-No'].isin(group_a)].copy()
n_remaining = len(remaining_df)
max_count_b = int(0.3 * n_remaining)
cumulative_sales_b = 0
group_b = []
for idx, row in remaining_df.iterrows():
    group_b.append(row['Series-No'])
    cumulative_sales_b += row['Total_Sales']
    if (cumulative_sales_b / overall_total >= 0.15) or (len(group_b) >= max_count_b):
        break

# Nhóm c: các series còn lại
group_c = remaining_df[~remaining_df['Series-No'].isin(group_b)]['Series-No'].tolist()

series_totals['Group'] = series_totals['Series-No'].apply(
    lambda x: 'a' if x in group_a else ('b' if x in group_b else 'c')
)

print("\nPhân nhóm doanh số (a, b, c):")
print("Tổng số Series:", n_total)
print("Nhóm a: {} Series, chiếm {:.2%} doanh số".format(len(group_a), cumulative_sales_a / overall_total))
print("Nhóm b: {} Series, chiếm {:.2%} doanh số".format(len(group_b), cumulative_sales_b / overall_total))
remaining_sales = overall_total - cumulative_sales_a - cumulative_sales_b
print("Nhóm c: {} Series, chiếm {:.2%} doanh số".format(len(group_c), remaining_sales / overall_total))






# -------------------------------
# Phần 3: Phân nhóm bổ sung x, y, z trong mỗi nhóm a, b, c theo hệ số biến động (CV)
# -------------------------------
# Tính CV từ dữ liệu 12 tháng cuối của wide format
last_12 = df_hierarchical.iloc[-12:]
def compute_cv(series_data):
    mean_val = series_data.mean()
    return np.nan if mean_val == 0 else series_data.std() / mean_val

# Tính CV cho tất cả các series
cv_dict = {series: compute_cv(last_12[series]) for series in series_cols}

# Áp dụng ngưỡng phân loại CV: X: CV < 0.34, Y: 0.34 ≤ CV < 0.84, Z: CV ≥ 0.84.
subgroup_dict = {}
for series, cv_val in cv_dict.items():
    if pd.isna(cv_val):
        subgroup = 'unknown'
    elif cv_val < 0.34:
        subgroup = 'x'
    elif cv_val < 0.84:
        subgroup = 'y'
    else:
        subgroup = 'z'
    subgroup_dict[series] = subgroup

# In bảng tóm tắt số lượng các nhóm phụ
from collections import Counter
subgroup_counts = Counter(subgroup_dict.values())
print("\nPhân loại phụ theo CV:")
for key in ['x', 'y', 'z', 'unknown']:
    print(f"Nhóm {key}: {subgroup_counts.get(key, 0)} series")

# -------------------------------
# Phần 4: Gán nhãn kết hợp vào tên cột wide format
# -------------------------------
# Tạo mapping cuối: tên cột sẽ có định dạng <Series-No>_<Group (a,b,c)>_<Subgroup (x,y,z,unknown)>
final_mapping = {}
for series in series_cols:
    # Lấy nhãn nhóm a,b,c từ series_totals
    group_label = series_totals.loc[series_totals['Series-No'] == series, 'Group'].values[0]
    subgroup_label = subgroup_dict.get(series, '')
    final_mapping[series] = f"{series}_{group_label}_{subgroup_label}"

# Cập nhật tên cột trong df_hierarchical (cột Total giữ nguyên)
new_columns = {}
for col in df_hierarchical.columns:
    if col == "Total":
        new_columns[col] = col
    else:
        new_columns[col] = final_mapping.get(col, col)

df_final = df_hierarchical.rename(columns=new_columns)

print("\nWide Format DataFrame sau khi gán nhãn a,b,c và x,y,z:")

df_final


Phân nhóm doanh số (a, b, c):
Tổng số Series: 1836
Nhóm a: 367 Series, chiếm 66.63% doanh số
Nhóm b: 254 Series, chiếm 15.00% doanh số
Nhóm c: 1215 Series, chiếm 18.37% doanh số

Phân loại phụ theo CV:
Nhóm x: 124 series
Nhóm y: 442 series
Nhóm z: 974 series
Nhóm unknown: 296 series

Wide Format DataFrame sau khi gán nhãn a,b,c và x,y,z:


,Total,36206_a_z,36207_a_unknown,36208_a_unknown,36229_a_z,36230_a_z,36232_a_z,53302_c_unknown,53303_a_unknown,A1000954_c_unknown,...,T505-106_a_z,T505-560_a_x,T505-562_b_z,T505-662_b_unknown,T505-742_b_unknown,T505-762_a_x,T505-962_b_unknown,T800_a_y,T800-111_a_z,T800-124_a_y
FiscalMonthLastDate,,,,,,,,,,,,,,,,,,,,,
2022-01-22,94220.0,272.0,370.0,442.0,120.0,145.0,135.0,18.0,158.0,15.0,...,92.0,243.0,109.0,122.0,99.0,286.0,80.0,184.0,339.0,193.0
2022-02-19,105423.0,272.0,479.0,590.0,122.0,173.0,282.0,37.0,213.0,6.0,...,119.0,227.0,97.0,125.0,128.0,262.0,120.0,183.0,327.0,179.0
2022-03-26,131285.0,277.0,562.0,573.0,267.0,274.0,343.0,15.0,226.0,19.0,...,109.0,234.0,151.0,87.0,117.0,314.0,115.0,275.0,369.0,740.0
2022-04-23,72222.0,165.0,198.0,320.0,104.0,98.0,143.0,27.0,120.0,1.0,...,117.0,160.0,90.0,149.0,77.0,223.0,83.0,171.0,312.0,326.0
2022-05-21,75760.0,157.0,201.0,231.0,162.0,144.0,144.0,16.0,140.0,2.0,...,76.0,144.0,76.0,194.0,79.0,164.0,81.0,155.0,253.0,127.0
2022-06-25,85249.0,153.0,213.0,351.0,181.0,168.0,200.0,11.0,144.0,4.0,...,96.0,177.0,84.0,69.0,68.0,342.0,71.0,250.0,454.0,473.0
2022-07-23,71975.0,111.0,175.0,201.0,115.0,177.0,179.0,20.0,157.0,6.0,...,78.0,164.0,50.0,58.0,90.0,488.0,43.0,144.0,87.0,77.0
2022-08-20,77834.0,120.0,255.0,370.0,109.0,70.0,169.0,3.0,165.0,1.0,...,71.0,275.0,99.0,228.0,95.0,335.0,113.0,116.0,112.0,52.0
2022-09-24,107000.0,232.0,222.0,368.0,166.0,192.0,246.0,6.0,283.0,1.0,...,834.0,277.0,151.0,134.0,178.0,707.0,141.0,197.0,452.0,100.0


# 4.Identify Stop Produced Series and Valid Series

In [ ]:
# -------------------------------
# Phần 5: Tách Stop Produced Series và Valid Series
# -------------------------------
# Lấy dữ liệu 12 tháng cuối
last_12 = df_final.iloc[-12:]

# Xác định các cột mà toàn bộ 12 giá trị cuối cùng đều bằng 0
stop_columns = [col for col in df_final.columns if (last_12[col] == 0).all()]

# Tạo DataFrame chứa các stop produced series và valid series
df_zero = df_final[stop_columns]
df_valid = df_final.drop(columns=stop_columns)

# In ra số lượng các series từng loại
print("Số lượng stop produced series:", len(stop_columns))
print("Số lượng valid series:", len(df_valid.columns) - (1 if 'Total' in df_valid.columns else 0)-1)

# -------------------------------
# Lưu kết quả thành 2 file CSV
# -------------------------------
df_zero.to_csv("Stopped_Series_Hierarchical.csv", index=True)
df_valid.to_csv("Series_Hierarchical.csv", index=True)


Số lượng stop produced series: 296
Số lượng valid series: 1539


#5.Use the Exponential Smoothing for Series x

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

# --- Bước 1: Load dữ liệu ---
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# --- Bước 2: Phân loại các series hợp lệ ---
# Loại bỏ cột "Total" và chỉ xét các cột chứa dữ liệu series
valid_series_cols = [col for col in df_final.columns if col != 'Total']

# Giả sử các cột đã được gán nhãn theo định dạng: <Series-No>_<group a,b,c>_<subgroup: x, y, z, unknown>
valid_x = [col for col in valid_series_cols if col.endswith('_x')]
valid_y = [col for col in valid_series_cols if col.endswith('_y')]
valid_z = [col for col in valid_series_cols if col.endswith('_z')]

print("Tổng số series hợp lệ:")
print("Nhóm x:", len(valid_x))
print("Nhóm y:", len(valid_y))
print("Nhóm z:", len(valid_z))

# --- Bước 3: Xây dựng hàm dự báo cho 1 series ---
def forecast_series(series, forecast_periods=6):
    """
    Chia dữ liệu thành train (all trừ forecast_periods cuối) và test (forecast_periods cuối),
    xây dựng mô hình Exponential Smoothing và trả về forecast cùng test set dưới dạng numpy array.
    Nếu không fit được mô hình, trả về None.
    """
    if len(series) < forecast_periods + 1:
        return None
    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]

    # Cố gắng fit mô hình với seasonality (chu kỳ 12 tháng) nếu có thể
    try:
        model = ExponentialSmoothing(train, trend='add', seasonal= None,
                                     seasonal_periods=12, damped_trend=True).fit()
    except Exception as e:
        try:
            model = ExponentialSmoothing(train, trend='add', seasonal=None,
                                         damped_trend=True).fit()
        except Exception as e2:
            return None
    forecast = model.forecast(forecast_periods)
    # Chuyển về mảng numpy để tránh lỗi căn chỉnh index
    return np.array(forecast), np.array(test)

# --- Bước 4: Hàm tính wMAPE cho 1 nhóm series ---
def compute_wmape(group, forecast_periods=6):
    total_abs_error = 0.0
    total_actual = 0.0
    valid_count = 0  # đếm số series có thể tính được wMAPE (không chia cho 0)
    for col in group:
        series = df_final[col]
        result = forecast_series(series, forecast_periods)
        if result is None:
            continue
        forecast_arr, test_arr = result
        # Nếu tổng giá trị thực của test = 0, bỏ qua series này để tránh chia cho 0
        if np.sum(test_arr) == 0:
            continue
        abs_err = np.abs(forecast_arr - test_arr)
        total_abs_error += np.sum(abs_err)
        total_actual += np.sum(test_arr)
        valid_count += 1
    if total_actual == 0:
        return np.nan, valid_count
    return total_abs_error / total_actual, valid_count

forecast_periods = 6
wmape_x, valid_count_x = compute_wmape(valid_x, forecast_periods)
wmape_y, valid_count_y = compute_wmape(valid_y, forecast_periods)
wmape_z, valid_count_z = compute_wmape(valid_z, forecast_periods)

# --- Bước 5: Tạo bảng tổng hợp kết quả ---
summary_data = {
    "Nhóm": ["x (CV < 0.34)", "y (0.34 ≤ CV < 0.84)", "z (CV ≥ 0.84)"],
    "Tổng số series hợp lệ": [len(valid_x), len(valid_y), len(valid_z)],
    "Số series tính được wMAPE": [valid_count_x, valid_count_y, valid_count_z],
    "wMAPE hiện tại": [
        f"{wmape_x:.2%}" if not np.isnan(wmape_x) else "NaN",
        f"{wmape_y:.2%}" if not np.isnan(wmape_y) else "NaN",
        f"{wmape_z:.2%}" if not np.isnan(wmape_z) else "NaN"
    ],
    "Yêu cầu wMAPE": ["< 25%", "< 30%", "< 35%"]
}

summary_df = pd.DataFrame(summary_data)
print("\nBảng tổng hợp:")
print(summary_df)

Tổng số series hợp lệ:
Nhóm x: 124
Nhóm y: 442
Nhóm z: 974

Bảng tổng hợp:
                   Nhóm  Tổng số series hợp lệ  Số series tính được wMAPE  \
0         x (CV < 0.34)                    124                        124   
1  y (0.34 ≤ CV < 0.84)                    442                        442   
2         z (CV ≥ 0.84)                    974                        853   

  wMAPE hiện tại Yêu cầu wMAPE  
0         29.47%         < 25%  
1         60.26%         < 30%  
2        116.84%         < 35%  


In [ ]:
summary_df

,Nhóm,Tổng số series hợp lệ,Số series tính được wMAPE,wMAPE hiện tại,Yêu cầu wMAPE
0,x (CV < 0.34),124,124,29.47%,< 25%
1,y (0.34 ≤ CV < 0.84),442,442,60.26%,< 30%
2,z (CV ≥ 0.84),974,853,116.84%,< 35%


# 6.Use the ARIMA/SARIMA & Exponential Smoothing for Group X. => Choose the model that have the best wMape for each series

## Code Block 1: Forecast Model cho Nhóm X (CV < 0.34 – dữ liệu ổn định)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# Load dữ liệu
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# Lọc các cột thuộc Group X (tên kết thúc bằng _x)
group_x_cols = [col for col in df_final.columns if col != 'Total' and col.endswith('_x')]
forecast_periods = 6
wmape_threshold = 0.25  # Yêu cầu: wMAPE < 25%

series_results = []
overall_total_abs_error = 0.0
overall_total_actual = 0.0

trend_threshold = 0.05  # Ngưỡng xác định xu hướng

# Các cấu hình ARIMA non-seasonal cần thử
non_seasonal_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1)]
# Các cấu hình SARIMA cần thử (non-seasonal_order, seasonal_order)
seasonal_orders = [((1,1,0), (1,0,1,12)), ((0,1,1), (1,0,1,12))]

for col in group_x_cols:
    series = df_final[col]
    if len(series) < forecast_periods + 1:
        series_results.append({
            "Series": col,
            "Model": None,
            "wMAPE": None,
            "Status": "Invalid: Not enough data"
        })
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    # Tính xu hướng: so sánh giá trị đầu và cuối của train
    initial = train.iloc[0]
    final = train.iloc[-1]
    trend_ratio = abs(final - initial) / (abs(initial) + 1e-8)

    candidate_models = []

    # Thử các mô hình ES
    if trend_ratio > trend_threshold:
        try:
            model_add = ExponentialSmoothing(train, trend='add', seasonal=None, damped_trend=True).fit()
            forecast_add = model_add.forecast(forecast_periods)
            candidate_models.append(("Holt's Add", model_add, np.array(forecast_add)))
        except Exception as e:
            pass
        if (train > 0).all():
            try:
                model_mult = ExponentialSmoothing(train, trend='mul', seasonal=None, damped_trend=True).fit()
                forecast_mult = model_mult.forecast(forecast_periods)
                candidate_models.append(("Holt's Mul", model_mult, np.array(forecast_mult)))
            except Exception as e:
                pass
    else:
        try:
            model_simple = ExponentialSmoothing(train, trend=None, seasonal=None).fit()
            forecast_simple = model_simple.forecast(forecast_periods)
            candidate_models.append(("Simple ES", model_simple, np.array(forecast_simple)))
        except Exception as e:
            pass

    # Chọn mô hình ES tốt nhất dựa trên tổng lỗi
    best_model_name = None
    best_error = np.inf
    best_forecast = None
    for name, model, forecast_arr in candidate_models:
        if np.sum(test_arr) == 0:
            continue
        error = np.sum(np.abs(forecast_arr - test_arr))
        if error < best_error:
            best_error = error
            best_forecast = forecast_arr
            best_model_name = name
    current_wmape = best_error / np.sum(test_arr) if np.sum(test_arr) > 0 else np.inf

    # Nếu wMAPE từ ES >= threshold, thử ARIMA grid search
    if current_wmape >= wmape_threshold:
        # Thử ARIMA non-seasonal
        for order in non_seasonal_orders:
            try:
                model_arima = ARIMA(train, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                forecast_arima = np.array(forecast_arima)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr)
                if wmape_arima < current_wmape:
                    best_model_name = f"ARIMA{order}"
                    best_error = error_arima
                    best_forecast = forecast_arima
                    current_wmape = wmape_arima
            except Exception as e:
                continue
        # Thử SARIMA nếu độ dài train đủ (> seasonal_period)
        seasonal_period = 12
        if len(train) > seasonal_period:
            for non_seasonal_order, seasonal_order in seasonal_orders:
                try:
                    model_sarima = SARIMAX(train, order=non_seasonal_order, seasonal_order=seasonal_order).fit(disp=False)
                    forecast_sarima = model_sarima.forecast(forecast_periods)
                    forecast_sarima = np.array(forecast_sarima)
                    error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                    wmape_sarima = error_sarima / np.sum(test_arr)
                    if wmape_sarima < current_wmape:
                        best_model_name = f"SARIMA{non_seasonal_order, seasonal_order}"
                        best_error = error_sarima
                        best_forecast = forecast_sarima
                        current_wmape = wmape_sarima
                except Exception as e:
                    continue

    if np.sum(test_arr) == 0:
        series_results.append({
            "Series": col,
            "Model": best_model_name,
            "wMAPE": None,
            "Status": "Invalid: Test sum zero"
        })
        continue

    status = "Valid" if current_wmape < wmape_threshold else "Invalid: wMAPE too high"

    if status == "Valid":
        overall_total_abs_error += best_error
        overall_total_actual += np.sum(test_arr)

    series_results.append({
        "Series": col,
        "Model": best_model_name,
        "wMAPE": current_wmape,
        "Status": status
    })

overall_wmape = overall_total_abs_error / overall_total_actual if overall_total_actual != 0 else np.nan

df_results = pd.DataFrame(series_results)
df_results["wMAPE (%)"] = df_results["wMAPE"].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)

summary = {
    "Group": "X (CV < 0.34)",
    "Total Series": len(group_x_cols),
    "Valid Series": df_results[df_results["Status"]=="Valid"].shape[0],
    "Invalid Series": df_results[df_results["Status"]!="Valid"].shape[0],
    "Overall wMAPE": f"{overall_wmape:.2%}" if pd.notnull(overall_wmape) else "NaN"
}
df_summary = pd.DataFrame([summary])

print("=== Group X - Individual Series Results (With Extended ARIMA/SARIMA Grid Search) ===")
print(df_results[["Series", "Model", "wMAPE (%)", "Status"]])
print("\n=== Group X - Summary (Valid series: wMAPE < 25%) ===")
df_summary

=== Group X - Individual Series Results (With Extended ARIMA/SARIMA Grid Search) ===
           Series                             Model wMAPE (%)  \
0    A2000108_a_x                    ARIMA(1, 1, 0)    22.78%   
1    A2000223_a_x                        Holt's Add    23.17%   
2    A2000255_a_x                        Holt's Add    28.40%   
3    A2000381_a_x                        Holt's Add    13.96%   
4    A2000412_a_x  SARIMA((1, 1, 0), (1, 0, 1, 12))    20.74%   
..            ...                               ...       ...   
119  A8010372_b_x  SARIMA((0, 1, 1), (1, 0, 1, 12))    21.71%   
120  A8010374_b_x                    ARIMA(2, 1, 1)    24.69%   
121  A8010378_b_x  SARIMA((1, 1, 0), (1, 0, 1, 12))    27.02%   
122  T505-560_a_x                        Holt's Mul    15.09%   
123  T505-762_a_x                    ARIMA(2, 1, 1)    26.23%   

                      Status  
0                      Valid  
1                      Valid  
2    Invalid: wMAPE too high  
3         

,Group,Total Series,Valid Series,Invalid Series,Overall wMAPE
0,X (CV < 0.34),124,75,49,19.54%


## Code Block 2: Forecast Model cho Nhóm Y (0.34 ≤ CV < 0.84 – dữ liệu có seasonality trung bình)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

# Load dữ liệu (nếu chưa load)
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# Lọc các cột thuộc nhóm Y (hậu tố _y)
group_y_cols = [col for col in df_final.columns if col != 'Total' and col.endswith('_y')]
forecast_periods = 6

results_y = {}
for col in group_y_cols:
    series = df_final[col]
    if len(series) < forecast_periods + 1:
        continue
    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    try:
        # Với dữ liệu có seasonality, dùng mô hình Holt-Winters với seasonal 'add' (giả sử chu kỳ 12 tháng)
        model = ExponentialSmoothing(train, trend='add', seasonal=None, seasonal_periods=12, damped_trend=True).fit()
        forecast = model.forecast(forecast_periods)
    except Exception as e:
        forecast = pd.Series([np.nan] * forecast_periods, index=test.index)
    results_y[col] = {"train": train, "test": test, "forecast": forecast}

# Tính wMAPE cho nhóm Y
total_abs_error = 0
total_actual = 0
for res in results_y.values():
    forecast_arr = res["forecast"].values
    test_arr = res["test"].values
    total_abs_error += np.sum(np.abs(forecast_arr - test_arr))
    total_actual += np.sum(test_arr)
wmape_y = total_abs_error / total_actual if total_actual != 0 else np.nan
print("Group Y - wMAPE: {:.2%}".format(wmape_y))

Group Y - wMAPE: 60.26%


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

warnings.filterwarnings("ignore")

# Load dữ liệu
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# Lọc các cột thuộc nhóm Y (tên kết thúc bằng _y)
group_y_cols = [col for col in df_final.columns if col != 'Total' and col.endswith('_y')]
forecast_periods = 6
wmape_threshold = 30  # Yêu cầu: wMAPE < 30%

series_results = []
overall_total_abs_error = 0.0
overall_total_actual = 0.0

trend_threshold = 0.05  # Ngưỡng xác định xu hướng

# Các cấu hình ARIMA non-seasonal cần thử
non_seasonal_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1)]
# Các cấu hình SARIMA cần thử (non-seasonal_order, seasonal_order)
seasonal_orders = [((1,1,0), (1,0,1,12)), ((0,1,1), (1,0,1,12))]

for col in group_y_cols:
    series = df_final[col].dropna()
    if len(series) < forecast_periods + 1:
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    # Tính xu hướng: so sánh giá trị đầu và cuối của train
    initial = train.iloc[0]
    final = train.iloc[-1]
    trend_ratio = abs(final - initial) / (abs(initial) + 1e-8)

    candidate_models = []

    # Thử các mô hình ES
    if trend_ratio > trend_threshold:
        try:
            model_add = ExponentialSmoothing(train, trend='add', seasonal=None, damped_trend=True).fit()
            forecast_add = model_add.forecast(forecast_periods)
            candidate_models.append(("Holt's Add", model_add, np.array(forecast_add)))
        except:
            pass
        if (train > 0).all():
            try:
                model_mult = ExponentialSmoothing(train, trend='mul', seasonal=None, damped_trend=True).fit()
                forecast_mult = model_mult.forecast(forecast_periods)
                candidate_models.append(("Holt's Mul", model_mult, np.array(forecast_mult)))
            except:
                pass
    else:
        try:
            model_simple = ExponentialSmoothing(train, trend=None, seasonal=None).fit()
            forecast_simple = model_simple.forecast(forecast_periods)
            candidate_models.append(("Simple ES", model_simple, np.array(forecast_simple)))
        except:
            pass

    # Chọn mô hình ES tốt nhất dựa trên tổng lỗi
    best_model_name = None
    best_error = np.inf
    best_forecast = None
    for name, model, forecast_arr in candidate_models:
        if np.sum(test_arr) == 0:
            continue
        error = np.sum(np.abs(forecast_arr - test_arr))
        if error < best_error:
            best_error = error
            best_forecast = forecast_arr
            best_model_name = name
    current_wmape = best_error / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

    # Nếu wMAPE từ ES >= threshold, thử ARIMA grid search
    if current_wmape >= wmape_threshold:
        # Thử ARIMA non-seasonal
        for order in non_seasonal_orders:
            try:
                model_arima = ARIMA(train, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                forecast_arima = np.array(forecast_arima)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr) * 100
                if wmape_arima < current_wmape:
                    best_model_name = f"ARIMA{order}"
                    best_error = error_arima
                    best_forecast = forecast_arima
                    current_wmape = wmape_arima
            except:
                continue

        # Thử SARIMA nếu độ dài train đủ (> seasonal_period)
        seasonal_period = 12
        if len(train) > seasonal_period:
            for non_seasonal_order, seasonal_order in seasonal_orders:
                try:
                    model_sarima = SARIMAX(train, order=non_seasonal_order, seasonal_order=seasonal_order).fit(disp=False)
                    forecast_sarima = model_sarima.forecast(forecast_periods)
                    forecast_sarima = np.array(forecast_sarima)
                    error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                    wmape_sarima = error_sarima / np.sum(test_arr) * 100
                    if wmape_sarima < current_wmape:
                        best_model_name = f"SARIMA{non_seasonal_order, seasonal_order}"
                        best_error = error_sarima
                        best_forecast = forecast_sarima
                        current_wmape = wmape_sarima
                except:
                    continue

    if np.sum(test_arr) == 0:
        continue

    status = "Valid" if current_wmape < wmape_threshold else "Invalid: wMAPE too high"

    if status == "Valid":
        overall_total_abs_error += best_error
        overall_total_actual += np.sum(test_arr)

    series_results.append({
        "Series": col,
        "Model": best_model_name,
        "wMAPE": current_wmape,
        "Status": status
    })

overall_wmape = overall_total_abs_error / overall_total_actual * 100 if overall_total_actual != 0 else np.nan

df_results = pd.DataFrame(series_results)
df_results_valid = df_results[df_results["Status"] == "Valid"]

# Hiển thị số lượng series hợp lệ
# Tạo summary cho nhóm Y
summary_y = {
    "Group": "Y (0.34 ≤ CV < 0.84)",
    "Total Series": len(group_y_cols),
    "Valid Series": df_results_valid.shape[0],
    "Invalid Series": df_results[df_results["Status"] != "Valid"].shape[0],
    "Overall wMAPE": f"{overall_wmape:.2f}%" if pd.notnull(overall_wmape) else "NaN"
}
df_summary_y = pd.DataFrame([summary_y])

# In kết quả từng series trong nhóm Y
print("=== Group Y - Individual Series Results (With Extended ARIMA/SARIMA Grid Search) ===")
print(df_results[["Series", "Model", "wMAPE", "Status"]])

# In summary của nhóm Y
print("\n=== Group Y - Summary (Valid series: wMAPE < 30%) ===")
print(df_summary_y)


=== Group Y - Individual Series Results (With Extended ARIMA/SARIMA Grid Search) ===
           Series                             Model       wMAPE  \
0    A2000094_a_y                        Holt's Add   28.099245   
1    A2000110_a_y                    ARIMA(1, 1, 1)   21.195140   
2    A2000125_a_y  SARIMA((1, 1, 0), (1, 0, 1, 12))   40.347075   
3    A2000130_a_y  SARIMA((0, 1, 1), (1, 0, 1, 12))   32.925142   
4    A2000133_a_y                        Holt's Mul   27.730483   
..            ...                               ...         ...   
437  A8010373_b_y  SARIMA((0, 1, 1), (1, 0, 1, 12))   39.096179   
438  A8010375_c_y  SARIMA((1, 1, 0), (1, 0, 1, 12))   46.003174   
439      B010_a_y                    ARIMA(2, 1, 1)   39.832553   
440      T800_a_y                    ARIMA(2, 1, 1)   82.293595   
441  T800-124_a_y                        Holt's Mul  157.838934   

                      Status  
0                      Valid  
1                      Valid  
2    Invalid: wMA

### Fine-tuning 1st time

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pandas.tseries.holiday import USFederalHolidayCalendar
import warnings

warnings.filterwarnings("ignore")

df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)



if df_final is not None:
    # Lọc các cột thuộc nhóm Y (hậu tố _y)
    group_y_cols = [col for col in df_final.columns if col != 'Total' and col.endswith('_y')]
    forecast_periods = 6
    wmape_threshold = 30  # Mục tiêu: cải thiện series có WMAPE < 30%

    # **Thêm holiday feature**
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df_final.index.min(), end=df_final.index.max())
    df_final["Holiday"] = df_final.index.isin(holidays).astype(int)

    fine_tuned_results = []

    print(f"🔄 Bắt đầu Fine-Tuning {len(group_y_cols)} series nhóm Y với holiday feature...")

    for col in group_y_cols:
        series = df_final[col].dropna()
        if len(series) < forecast_periods + 1:
            continue

        train = series.iloc[:-forecast_periods]
        test = series.iloc[-forecast_periods:]
        test_arr = np.array(test)

        train_holiday = df_final["Holiday"].iloc[:-forecast_periods]
        test_holiday = df_final["Holiday"].iloc[-forecast_periods:]

        # 🔹 Fine-tune Holt-Winters với nhiều thông số
        best_wmape = np.inf
        best_model = None
        best_forecast = None
        best_params = None

        hw_configs = [
            ("add", "add"), ("add", "mul"), ("mul", "add"), ("mul", "mul")
        ]

        for trend, seasonal in hw_configs:
            try:
                model_hw = ExponentialSmoothing(
                    train, trend=trend, seasonal=seasonal,
                    seasonal_periods=12, damped_trend=True
                ).fit()
                forecast_hw = model_hw.forecast(forecast_periods)
                error_hw = np.sum(np.abs(forecast_hw - test_arr))
                wmape_hw = error_hw / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

                if wmape_hw < best_wmape:
                    best_wmape = wmape_hw
                    best_model = model_hw
                    best_forecast = forecast_hw
                    best_params = f"Holt-Winters({trend}, {seasonal})"
            except:
                continue

        # 🔹 Fine-tune ARIMA nếu Holt-Winters chưa tốt
        if best_wmape >= wmape_threshold:
            arima_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1)]
            for order in arima_orders:
                try:
                    model_arima = ARIMA(train, order=order).fit()
                    forecast_arima = model_arima.forecast(forecast_periods)
                    error_arima = np.sum(np.abs(forecast_arima - test_arr))
                    wmape_arima = error_arima / np.sum(test_arr) * 100

                    if wmape_arima < best_wmape:
                        best_wmape = wmape_arima
                        best_model = model_arima
                        best_forecast = forecast_arima
                        best_params = f"ARIMA{order}"
                except:
                    continue

        # 🔹 Fine-tune SARIMA nếu ARIMA chưa tốt
        if best_wmape >= wmape_threshold:
            sarima_orders = [((1,1,0), (1,0,1,12)), ((0,1,1), (1,0,1,12))]
            for non_seasonal, seasonal in sarima_orders:
                try:
                    model_sarima = SARIMAX(train, order=non_seasonal, seasonal_order=seasonal).fit(disp=False)
                    forecast_sarima = model_sarima.forecast(forecast_periods)
                    error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                    wmape_sarima = error_sarima / np.sum(test_arr) * 100

                    if wmape_sarima < best_wmape:
                        best_wmape = wmape_sarima
                        best_model = model_sarima
                        best_forecast = forecast_sarima
                        best_params = f"SARIMA{non_seasonal, seasonal}"
                except:
                    continue

        # 🔹 Lưu kết quả fine-tuning
        fine_tuned_results.append({
            "Series": col,
            "Best Model": best_params,
            "New WMAPE": best_wmape,
            "Status": "Improved" if best_wmape < wmape_threshold else "Not Improved"
        })

    # **Hiển thị kết quả fine-tuning**
    df_fine_tuned = pd.DataFrame(fine_tuned_results)
    print("\n=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===")
    print(df_fine_tuned.to_string(index=False))

    # Hiển thị số lượng series cải thiện được
    num_improved = df_fine_tuned[df_fine_tuned["Status"] == "Improved"].shape[0]
    print(f"\n✅ Số series được cải thiện sau fine-tuning: {num_improved}/{len(group_y_cols)}")


🔄 Bắt đầu Fine-Tuning 442 series nhóm Y với holiday feature...

=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===
       Series                       Best Model  New WMAPE       Status
 A2000094_a_y           Holt-Winters(add, mul)  28.792905     Improved
 A2000110_a_y                   ARIMA(1, 1, 1)  21.195140     Improved
 A2000125_a_y           Holt-Winters(add, add)  34.577791 Not Improved
 A2000130_a_y           Holt-Winters(add, add)  23.559750     Improved
 A2000133_a_y           Holt-Winters(mul, add)  26.528556     Improved
 A2000135_a_y SARIMA((1, 1, 0), (1, 0, 1, 12))  49.357693 Not Improved
 A2000136_a_y                   ARIMA(1, 1, 1)  85.328354 Not Improved
 A2000249_a_y                   ARIMA(0, 1, 1)  27.628362     Improved
 A2000329_a_y           Holt-Winters(mul, mul)  41.730152 Not Improved
 A2000346_a_y                   ARIMA(1, 1, 0)  64.366619 Not Improved
 A2000348_a_y           Holt-Winters(add, mul)  18.791719     Improved
 

### Fine-tuning 2nd time

In [ ]:
# 🔄 Fine-tuning lại các series chưa cải thiện bằng phương pháp nâng cao
print("\n🔄 Bắt đầu Fine-Tuning lại các series chưa đạt WMAPE < 30%...")

# **Lọc ra danh sách các series chưa đạt yêu cầu**
df_not_improved = df_fine_tuned[df_fine_tuned["Status"] == "Not Improved"]
series_to_finetune = df_not_improved["Series"].tolist()

# **Lưu kết quả fine-tuning nâng cao**
fine_tuned_results_v2 = []

for col in series_to_finetune:
    series = df_final[col].dropna()
    if len(series) < forecast_periods + 1:
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    train_holiday = df_final["Holiday"].iloc[:-forecast_periods]
    test_holiday = df_final["Holiday"].iloc[-forecast_periods:]

    # **🔹 Bổ sung Moving Average để làm trơn dữ liệu**
    train_smooth = train.rolling(window=3, min_periods=1).mean()

    # **🔹 Thử Holt-Winters với thông số mở rộng hơn**
    best_wmape = np.inf
    best_model = None
    best_forecast = None
    best_params = None

    hw_configs = [
        ("add", "add"), ("add", "mul"), ("mul", "add"), ("mul", "mul")
    ]

    for trend, seasonal in hw_configs:
        try:
            model_hw = ExponentialSmoothing(
                train_smooth, trend=trend, seasonal=seasonal,
                seasonal_periods=12, damped_trend=True
            ).fit()
            forecast_hw = model_hw.forecast(forecast_periods)
            error_hw = np.sum(np.abs(forecast_hw - test_arr))
            wmape_hw = error_hw / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

            if wmape_hw < best_wmape:
                best_wmape = wmape_hw
                best_model = model_hw
                best_forecast = forecast_hw
                best_params = f"Holt-Winters({trend}, {seasonal})"
        except:
            continue

    # **🔹 Thử lại ARIMA/SARIMA nếu Holt-Winters chưa tốt**
    if best_wmape >= wmape_threshold:
        arima_orders = [(2,1,2), (1,1,2), (2,1,1), (3,1,2)]
        for order in arima_orders:
            try:
                model_arima = ARIMA(train, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr) * 100

                if wmape_arima < best_wmape:
                    best_wmape = wmape_arima
                    best_model = model_arima
                    best_forecast = forecast_arima
                    best_params = f"ARIMA{order}"
            except:
                continue

    # **🔹 Thử SARIMA nếu ARIMA chưa tốt**
    if best_wmape >= wmape_threshold:
        sarima_orders = [((2,1,1), (1,0,1,12)), ((1,1,2), (1,0,1,12))]
        for non_seasonal, seasonal in sarima_orders:
            try:
                model_sarima = SARIMAX(train, order=non_seasonal, seasonal_order=seasonal).fit(disp=False)
                forecast_sarima = model_sarima.forecast(forecast_periods)
                error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                wmape_sarima = error_sarima / np.sum(test_arr) * 100

                if wmape_sarima < best_wmape:
                    best_wmape = wmape_sarima
                    best_model = model_sarima
                    best_forecast = forecast_sarima
                    best_params = f"SARIMA{non_seasonal, seasonal}"
            except:
                continue

    # **🔹 Lưu kết quả fine-tuning nâng cao**
    fine_tuned_results_v2.append({
        "Series": col,
        "Best Model": best_params,
        "New WMAPE": best_wmape,
        "Status": "Improved" if best_wmape < wmape_threshold else "Not Improved"
    })

# **Hiển thị kết quả fine-tuning nâng cao**
df_fine_tuned_v2 = pd.DataFrame(fine_tuned_results_v2)
print("\n=== 🔥 Fine-Tuning Lần 2: Cải Thiện Các Series Chưa Đạt ===")
print(df_fine_tuned_v2.to_string(index=False))

# **Tính số lượng series được cải thiện sau fine-tuning lần 2**
num_improved_v2 = df_fine_tuned_v2[df_fine_tuned_v2["Status"] == "Improved"].shape[0]
print(f"\n✅ Số series được cải thiện sau Fine-Tuning lần 2: {num_improved_v2}/{len(series_to_finetune)}")



🔄 Bắt đầu Fine-Tuning lại các series chưa đạt WMAPE < 30%...

=== 🔥 Fine-Tuning Lần 2: Cải Thiện Các Series Chưa Đạt ===
       Series                       Best Model  New WMAPE       Status
 A2000125_a_y           Holt-Winters(mul, add)   0.000000     Improved
 A2000135_a_y           Holt-Winters(add, add)  77.930792 Not Improved
 A2000136_a_y           Holt-Winters(mul, add)  60.792446 Not Improved
 A2000329_a_y           Holt-Winters(add, add)  37.387203 Not Improved
 A2000346_a_y                   ARIMA(2, 1, 2)  64.042950 Not Improved
 A2000355_a_y           Holt-Winters(mul, mul)  55.882914 Not Improved
 A2000370_a_y           Holt-Winters(mul, mul)  46.922697 Not Improved
 A2000388_a_y SARIMA((2, 1, 1), (1, 0, 1, 12))  62.098984 Not Improved
 A2000407_a_y           Holt-Winters(mul, add)   0.000000     Improved
 A2000409_b_y           Holt-Winters(add, mul)  95.228778 Not Improved
 A2000410_a_y           Holt-Winters(add, add)  64.774062 Not Improved
 A2000416_a_y           Ho

### Fine-tuning 3rd time

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# 🔄 Fine-tuning lần 3 với các phương pháp nâng cao (Không dùng Prophet)
print("\n🔄 Bắt đầu Fine-Tuning lần 3 cho các series chưa đạt WMAPE < 30%...")

# **Lọc ra danh sách các series chưa đạt yêu cầu từ lần 2**
df_not_improved_v2 = df_fine_tuned_v2[df_fine_tuned_v2["Status"] == "Not Improved"]
series_to_finetune_v3 = df_not_improved_v2["Series"].tolist()

# **Lưu kết quả fine-tuning nâng cao lần 3**
fine_tuned_results_v3 = []

for col in series_to_finetune_v3:
    series = df_final[col].dropna()
    if len(series) < forecast_periods + 1:
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    # **🔹 Sử dụng Differencing để loại bỏ xu hướng**
    train_diff = train.diff().dropna()

    # **🔹 Holt-Winters với period dài hơn**
    best_wmape = np.inf
    best_model = None
    best_forecast = None
    best_params = None

    hw_configs = [12, 18, 24]  # Thử nhiều seasonal period hơn
    for period in hw_configs:
        try:
            model_hw = ExponentialSmoothing(
                train, trend='add', seasonal='add',
                seasonal_periods=period, damped_trend=True
            ).fit()
            forecast_hw = model_hw.forecast(forecast_periods)
            error_hw = np.sum(np.abs(forecast_hw - test_arr))
            wmape_hw = error_hw / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

            if wmape_hw < best_wmape:
                best_wmape = wmape_hw
                best_model = model_hw
                best_forecast = forecast_hw
                best_params = f"Holt-Winters({period})"
        except:
            continue

    # **🔹 Thử ARIMA mở rộng nếu Holt-Winters chưa tốt**
    if best_wmape >= wmape_threshold:
        arima_orders = [(2,1,2), (1,1,2), (2,1,1), (3,1,2)]
        for order in arima_orders:
            try:
                model_arima = ARIMA(train, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr) * 100

                if wmape_arima < best_wmape:
                    best_wmape = wmape_arima
                    best_model = model_arima
                    best_forecast = forecast_arima
                    best_params = f"ARIMA{order}"
            except:
                continue

    # **🔹 Thử SARIMA nếu ARIMA chưa tốt**
    if best_wmape >= wmape_threshold:
        sarima_orders = [((2,1,1), (1,0,1,12)), ((1,1,2), (1,0,1,12))]
        for non_seasonal, seasonal in sarima_orders:
            try:
                model_sarima = SARIMAX(train, order=non_seasonal, seasonal_order=seasonal).fit(disp=False)
                forecast_sarima = model_sarima.forecast(forecast_periods)
                error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                wmape_sarima = error_sarima / np.sum(test_arr) * 100

                if wmape_sarima < best_wmape:
                    best_wmape = wmape_sarima
                    best_model = model_sarima
                    best_forecast = forecast_sarima
                    best_params = f"SARIMA{non_seasonal, seasonal}"
            except:
                continue

    # **🔹 Thử XGBoost Regressor nếu SARIMA chưa tốt**
    if best_wmape >= wmape_threshold:
        try:
            # Chuẩn bị dữ liệu cho XGBoost
            x_train = np.arange(len(train)).reshape(-1, 1)
            y_train = train.values
            x_test = np.arange(len(train), len(train) + forecast_periods).reshape(-1, 1)

            model_xgb = GradientBoostingRegressor()
            model_xgb.fit(x_train, y_train)
            forecast_xgb = model_xgb.predict(x_test)

            error_xgb = np.sum(np.abs(forecast_xgb - test_arr))
            wmape_xgb = error_xgb / np.sum(test_arr) * 100

            if wmape_xgb < best_wmape:
                best_wmape = wmape_xgb
                best_model = model_xgb
                best_forecast = forecast_xgb
                best_params = "XGBoost Regressor"
        except:
            pass

    # **🔹 Lưu kết quả fine-tuning lần 3**
    fine_tuned_results_v3.append({
        "Series": col,
        "Best Model": best_params,
        "New WMAPE": best_wmape,
        "Status": "Improved" if best_wmape < wmape_threshold else "Not Improved"
    })

# **Hiển thị kết quả fine-tuning lần 3**
df_fine_tuned_v3 = pd.DataFrame(fine_tuned_results_v3)
print("\n=== 🔥 Fine-Tuning Lần 3: Cải Thiện Các Series Chưa Đạt ===")
print(df_fine_tuned_v3.to_string(index=False))

# **Tính số lượng series được cải thiện sau fine-tuning lần 3**
num_improved_v3 = df_fine_tuned_v3[df_fine_tuned_v3["Status"] == "Improved"].shape[0]
print(f"\n✅ Số series được cải thiện sau Fine-Tuning lần 3: {num_improved_v3}/{len(series_to_finetune_v3)}")



🔄 Bắt đầu Fine-Tuning lần 3 cho các series chưa đạt WMAPE < 30%...

=== 🔥 Fine-Tuning Lần 3: Cải Thiện Các Series Chưa Đạt ===
       Series                       Best Model  New WMAPE       Status
 A2000135_a_y                XGBoost Regressor  71.735342 Not Improved
 A2000136_a_y                   ARIMA(1, 1, 2)  87.220188 Not Improved
 A2000329_a_y                   ARIMA(3, 1, 2)  43.843712 Not Improved
 A2000346_a_y                XGBoost Regressor  60.789367 Not Improved
 A2000355_a_y                 Holt-Winters(12)  48.044960 Not Improved
 A2000370_a_y                 Holt-Winters(12)  40.815270 Not Improved
 A2000388_a_y SARIMA((2, 1, 1), (1, 0, 1, 12))  62.098984 Not Improved
 A2000409_b_y                XGBoost Regressor 218.355563 Not Improved
 A2000410_a_y SARIMA((1, 1, 2), (1, 0, 1, 12))  71.899733 Not Improved
 A2000416_a_y                 Holt-Winters(12)  42.178256 Not Improved
 A2000442_a_y                XGBoost Regressor  38.560000 Not Improved
 A2000461_a_y       

### Fine-tuning 4th time

In [ ]:
# Kiểm tra xem thư viện Prophet có sẵn không
try:
    from prophet import Prophet
    prophet_available = True
except ImportError:
    prophet_available = False

# Kiểm tra xem TensorFlow/Keras có sẵn không (để chạy LSTM)
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from sklearn.preprocessing import MinMaxScaler
    lstm_available = True
except ImportError:
    lstm_available = False

# Báo cáo lại kết quả kiểm tra
if prophet_available and lstm_available:
    print("✅ Cả Prophet và LSTM đều khả dụng! Sẽ chạy cả hai mô hình.")
elif prophet_available:
    print("✅ Prophet khả dụng! Sẽ chỉ chạy Prophet.")
elif lstm_available:
    print("✅ LSTM khả dụng! Sẽ chỉ chạy LSTM.")
else:
    print("⚠️ Cả Prophet và LSTM đều không khả dụng! Không thể tiếp tục fine-tuning với mô hình này.")


✅ Cả Prophet và LSTM đều khả dụng! Sẽ chạy cả hai mô hình.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# Load dữ liệu
df_final = pd.read_csv("Series_Hierarchical.csv", index_col=0, parse_dates=True)

# Kiểm tra nếu df_fine_tuned_v3 có tồn tại
if 'df_fine_tuned_v3' not in locals() or df_fine_tuned_v3 is None:
    df_fine_tuned_v3 = pd.DataFrame(columns=["Series", "Best Model", "New WMAPE", "Status"])

# Kiểm tra cột "Status"
if "Status" not in df_fine_tuned_v3.columns:
    df_fine_tuned_v3["Status"] = "Not Improved"

# Lọc các series chưa đạt yêu cầu
df_not_improved = df_fine_tuned_v3[df_fine_tuned_v3["Status"] == "Not Improved"]
series_to_finetune = df_not_improved["Series"].tolist()

fine_tuned_results_v4 = []

for col in series_to_finetune:
    series = df_final[col].dropna()

    # **Kiểm tra nếu dữ liệu quá ít (< 6 tháng) thì bỏ qua**
    if len(series) < 6:
        print(f"⚠️ Series {col} có quá ít dữ liệu, bỏ qua!")
        continue

    train = series.iloc[:-6]
    test = series.iloc[-6:]

    best_wmape = np.inf
    best_model = None
    best_forecast = None

    # **1️⃣ Auto ARIMA**
    try:
        model_arima = auto_arima(train, seasonal=False, stepwise=True, suppress_warnings=True)
        forecast_arima = model_arima.predict(n_periods=6)
        wmape_arima = sum(abs(forecast_arima - test)) / sum(test) * 100

        if wmape_arima < best_wmape:
            best_wmape = wmape_arima
            best_model = "Auto ARIMA"
            best_forecast = forecast_arima
    except:
        pass

    # **2️⃣ Holt-Winters**
    try:
        model_hw = ExponentialSmoothing(train, trend='add', seasonal=None, damped_trend=True).fit()
        forecast_hw = model_hw.forecast(6)
        wmape_hw = sum(abs(forecast_hw - test)) / sum(test) * 100

        if wmape_hw < best_wmape:
            best_wmape = wmape_hw
            best_model = "Holt-Winters"
            best_forecast = forecast_hw
    except:
        pass

    # **3️⃣ Random Forest Regressor**
    try:
        scaler = MinMaxScaler(feature_range=(0, 1))
        train_scaled = scaler.fit_transform(train.values.reshape(-1, 1))

        X_train = np.arange(len(train_scaled)).reshape(-1, 1)
        y_train = train_scaled.flatten()
        X_test = np.arange(len(train_scaled), len(train_scaled) + 6).reshape(-1, 1)

        model_rf = RandomForestRegressor(n_estimators=50)
        model_rf.fit(X_train, y_train)
        forecast_rf = model_rf.predict(X_test)
        forecast_rf = scaler.inverse_transform(forecast_rf.reshape(-1, 1)).flatten()

        wmape_rf = sum(abs(forecast_rf - test)) / sum(test) * 100

        if wmape_rf < best_wmape:
            best_wmape = wmape_rf
            best_model = "Random Forest"
            best_forecast = forecast_rf
    except:
        pass

    # **4️⃣ Kết hợp Holt-Winters + XGBoost**
    try:
        from xgboost import XGBRegressor

        model_hw = ExponentialSmoothing(train, trend='add', seasonal=None, damped_trend=True).fit()
        residuals = train - model_hw.fittedvalues

        X_train = np.arange(len(residuals)).reshape(-1, 1)
        y_train = residuals.values
        X_test = np.arange(len(residuals), len(residuals) + 6).reshape(-1, 1)

        model_xgb = XGBRegressor()
        model_xgb.fit(X_train, y_train)
        forecast_xgb = model_xgb.predict(X_test) + model_hw.forecast(6)

        wmape_xgb = sum(abs(forecast_xgb - test)) / sum(test) * 100

        if wmape_xgb < best_wmape:
            best_wmape = wmape_xgb
            best_model = "Holt-Winters + XGBoost"
            best_forecast = forecast_xgb
    except:
        pass

    # **Lưu kết quả**
    fine_tuned_results_v4.append({
        "Series": col,
        "Best Model": best_model,
        "New WMAPE": best_wmape,
        "Status": "Improved" if best_wmape < 30 else "Not Improved"
    })

# **Nếu không có series nào được train**
if len(fine_tuned_results_v4) == 0:
    print("⚠️ Không có series nào đủ dữ liệu để train Auto ARIMA, Holt-Winters, Random Forest hoặc XGBoost.")
else:
    # **Hiển thị kết quả Fine-Tuning**
    df_fine_tuned_v4 = pd.DataFrame(fine_tuned_results_v4)
    print("\n=== 🔥 Fine-Tuning Lần 4 (Auto ARIMA + Holt-Winters + Random Forest + XGBoost) ===")
    print(df_fine_tuned_v4.to_string(index=False))

    # **Số series được cải thiện**
    num_improved_v4 = df_fine_tuned_v4[df_fine_tuned_v4["Status"] == "Improved"].shape[0]
    print(f"\n✅ Số series được cải thiện: {num_improved_v4}/{len(series_to_finetune)}")



=== 🔥 Fine-Tuning Lần 4 (Auto ARIMA + Holt-Winters + Random Forest + XGBoost) ===
       Series    Best Model  New WMAPE       Status
 A2000135_a_y Random Forest  53.490909 Not Improved
 A2000136_a_y Random Forest 100.310178 Not Improved
 A2000329_a_y Random Forest  55.628866 Not Improved
 A2000346_a_y Random Forest  62.388824 Not Improved
 A2000355_a_y Random Forest 113.664286 Not Improved
 A2000370_a_y Random Forest  54.823529 Not Improved
 A2000388_a_y Random Forest  74.354680 Not Improved
 A2000409_b_y Random Forest 224.266667 Not Improved
 A2000410_a_y Random Forest 108.528302 Not Improved
 A2000416_a_y Random Forest  46.656700 Not Improved
 A2000442_a_y Random Forest  38.560000 Not Improved
 A2000461_a_y Random Forest  35.305660 Not Improved
 A2000484_b_y Random Forest  46.014388 Not Improved
 A2000485_a_y Random Forest  37.423313 Not Improved
 A2000488_a_y Random Forest  41.676471 Not Improved
 A2000490_b_y Random Forest  49.152542 Not Improved
 A2000491_b_y Random Forest  51.3

In [ ]:
!pip install pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.6 MB/s eta 0:00:00


### Fine-tuning 5th time

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV

# Load dữ liệu
df_final = pd.read_csv("Series_Hierarchical.csv", index_col=0, parse_dates=True)

# Kiểm tra nếu df_fine_tuned_v4 có tồn tại
if 'df_fine_tuned_v4' not in locals() or df_fine_tuned_v4 is None:
    df_fine_tuned_v4 = pd.DataFrame(columns=["Series", "Best Model", "New WMAPE", "Status"])

# Kiểm tra cột "Status"
if "Status" not in df_fine_tuned_v4.columns:
    df_fine_tuned_v4["Status"] = "Not Improved"

# Lọc các series chưa đạt yêu cầu
df_not_improved = df_fine_tuned_v4[df_fine_tuned_v4["Status"] == "Not Improved"]
series_to_finetune = df_not_improved["Series"].tolist()

fine_tuned_results_v5 = []

for col in series_to_finetune:
    series = df_final[col].dropna()

    # **Kiểm tra nếu dữ liệu quá ít (< 6 tháng) thì bỏ qua**
    if len(series) < 6:
        print(f"⚠️ Series {col} có quá ít dữ liệu, bỏ qua!")
        continue

    train = series.iloc[:-6]
    test = series.iloc[-6:]

    best_wmape = np.inf
    best_model = None
    best_forecast = None

    # **1️⃣ Holt-Winters (Tối ưu lại)**
    for trend in ['add', 'mul']:
        for seasonal in ['add', 'mul']:
            try:
                model_hw = ExponentialSmoothing(train, trend=trend, seasonal=seasonal, seasonal_periods=6, damped_trend=True).fit()
                forecast_hw = model_hw.forecast(6)
                wmape_hw = sum(abs(forecast_hw - test)) / sum(test) * 100

                if wmape_hw < best_wmape:
                    best_wmape = wmape_hw
                    best_model = f"Holt-Winters ({trend}, {seasonal})"
                    best_forecast = forecast_hw
            except:
                pass

    # **2️⃣ ARIMA với nhiều tham số**
    arima_params = [(1,1,0), (0,1,1), (1,1,1), (2,1,1), (2,1,2)]
    for params in arima_params:
        try:
            model_arima = ARIMA(train, order=params).fit()
            forecast_arima = model_arima.forecast(6)
            wmape_arima = sum(abs(forecast_arima - test)) / sum(test) * 100

            if wmape_arima < best_wmape:
                best_wmape = wmape_arima
                best_model = f"ARIMA {params}"
                best_forecast = forecast_arima
        except:
            pass

    # **3️⃣ Random Forest (Grid Search Tuning)**
    try:
        scaler = MinMaxScaler(feature_range=(0, 1))
        train_scaled = scaler.fit_transform(train.values.reshape(-1, 1))

        X_train = np.arange(len(train_scaled)).reshape(-1, 1)
        y_train = train_scaled.flatten()
        X_test = np.arange(len(train_scaled), len(train_scaled) + 6).reshape(-1, 1)

        param_grid_rf = {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5, 10]
        }

        model_rf = RandomForestRegressor()
        grid_search_rf = GridSearchCV(model_rf, param_grid_rf, cv=3, n_jobs=-1)
        grid_search_rf.fit(X_train, y_train)

        best_rf = grid_search_rf.best_estimator_
        forecast_rf = best_rf.predict(X_test)
        forecast_rf = scaler.inverse_transform(forecast_rf.reshape(-1, 1)).flatten()

        wmape_rf = sum(abs(forecast_rf - test)) / sum(test) * 100

        if wmape_rf < best_wmape:
            best_wmape = wmape_rf
            best_model = f"Random Forest (Tuned)"
            best_forecast = forecast_rf
    except:
        pass

    # **4️⃣ XGBoost (Grid Search Tuning)**
    try:
        param_grid_xgb = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7]
        }

        model_xgb = XGBRegressor()
        grid_search_xgb = GridSearchCV(model_xgb, param_grid_xgb, cv=3, n_jobs=-1)
        grid_search_xgb.fit(X_train, y_train)

        best_xgb = grid_search_xgb.best_estimator_
        forecast_xgb = best_xgb.predict(X_test)
        forecast_xgb = scaler.inverse_transform(forecast_xgb.reshape(-1, 1)).flatten()

        wmape_xgb = sum(abs(forecast_xgb - test)) / sum(test) * 100

        if wmape_xgb < best_wmape:
            best_wmape = wmape_xgb
            best_model = f"XGBoost (Tuned)"
            best_forecast = forecast_xgb
    except:
        pass

    # **Lưu kết quả**
    fine_tuned_results_v5.append({
        "Series": col,
        "Best Model": best_model,
        "New WMAPE": best_wmape,
        "Status": "Improved" if best_wmape < 30 else "Not Improved"
    })

# **Hiển thị kết quả Fine-Tuning Lần 5**
df_fine_tuned_v5 = pd.DataFrame(fine_tuned_results_v5)
print("\n=== 🔥 Fine-Tuning Lần 5 (Holt-Winters + ARIMA + RF + XGBoost) ===")
print(df_fine_tuned_v5.to_string(index=False))

# **Số series được cải thiện**
num_improved_v5 = df_fine_tuned_v5[df_fine_tuned_v5["Status"] == "Improved"].shape[0]
print(f"\n✅ Số series được cải thiện: {num_improved_v5}/{len(series_to_finetune)}")



=== 🔥 Fine-Tuning Lần 5 (Holt-Winters + ARIMA + RF + XGBoost) ===
       Series            Best Model  New WMAPE       Status
 A2000135_a_y       XGBoost (Tuned)  94.923651 Not Improved
 A2000136_a_y Random Forest (Tuned) 107.525207 Not Improved
 A2000329_a_y Random Forest (Tuned)  55.059794 Not Improved
 A2000346_a_y Random Forest (Tuned)  67.907900 Not Improved
 A2000355_a_y Random Forest (Tuned) 104.299395 Not Improved
 A2000370_a_y Random Forest (Tuned)  56.688644 Not Improved
 A2000388_a_y Random Forest (Tuned)  73.187173 Not Improved
 A2000409_b_y       XGBoost (Tuned) 218.805021 Not Improved
 A2000410_a_y       XGBoost (Tuned)  96.638510 Not Improved
 A2000416_a_y Random Forest (Tuned)  49.000831 Not Improved
 A2000442_a_y       XGBoost (Tuned)  46.234580 Not Improved
 A2000461_a_y Random Forest (Tuned)  34.158491 Not Improved
 A2000484_b_y Random Forest (Tuned)  41.280576 Not Improved
 A2000485_a_y Random Forest (Tuned)  37.423313 Not Improved
 A2000488_a_y       XGBoost (Tune

In [ ]:
!pip install scikit-optimize

## Code Block 3: Forecast Model cho Nhóm Z z (CV ≥ 0.84)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

# Load dữ liệu
df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)

# Lọc các cột thuộc Group X (tên kết thúc bằng _x)
group_x_cols = [col for col in df_final.columns if col != 'Total' and col.endswith('_z')]
forecast_periods = 6
wmape_threshold = 0.35  # Yêu cầu: wMAPE < 25%

series_results = []
overall_total_abs_error = 0.0
overall_total_actual = 0.0

trend_threshold = 0.05  # Ngưỡng xác định xu hướng

# Các cấu hình ARIMA non-seasonal cần thử
non_seasonal_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1)]
# Các cấu hình SARIMA cần thử (non-seasonal_order, seasonal_order)
seasonal_orders = [((1,1,0), (1,0,1,12)), ((0,1,1), (1,0,1,12))]

for col in group_x_cols:
    series = df_final[col]
    if len(series) < forecast_periods + 1:
        series_results.append({
            "Series": col,
            "Model": None,
            "wMAPE": None,
            "Status": "Invalid: Not enough data"
        })
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    # Tính xu hướng: so sánh giá trị đầu và cuối của train
    initial = train.iloc[0]
    final = train.iloc[-1]
    trend_ratio = abs(final - initial) / (abs(initial) + 1e-8)

    candidate_models = []

    # Thử các mô hình ES
    if trend_ratio > trend_threshold:
        try:
            model_add = ExponentialSmoothing(train, trend='add', seasonal=None, damped_trend=True).fit()
            forecast_add = model_add.forecast(forecast_periods)
            candidate_models.append(("Holt's Add", model_add, np.array(forecast_add)))
        except Exception as e:
            pass
        if (train > 0).all():
            try:
                model_mult = ExponentialSmoothing(train, trend='mul', seasonal=None, damped_trend=True).fit()
                forecast_mult = model_mult.forecast(forecast_periods)
                candidate_models.append(("Holt's Mul", model_mult, np.array(forecast_mult)))
            except Exception as e:
                pass
    else:
        try:
            model_simple = ExponentialSmoothing(train, trend=None, seasonal=None).fit()
            forecast_simple = model_simple.forecast(forecast_periods)
            candidate_models.append(("Simple ES", model_simple, np.array(forecast_simple)))
        except Exception as e:
            pass

    # Chọn mô hình ES tốt nhất dựa trên tổng lỗi
    best_model_name = None
    best_error = np.inf
    best_forecast = None
    for name, model, forecast_arr in candidate_models:
        if np.sum(test_arr) == 0:
            continue
        error = np.sum(np.abs(forecast_arr - test_arr))
        if error < best_error:
            best_error = error
            best_forecast = forecast_arr
            best_model_name = name
    current_wmape = best_error / np.sum(test_arr) if np.sum(test_arr) > 0 else np.inf

    # Nếu wMAPE từ ES >= threshold, thử ARIMA grid search
    if current_wmape >= wmape_threshold:
        # Thử ARIMA non-seasonal
        for order in non_seasonal_orders:
            try:
                model_arima = ARIMA(train, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                forecast_arima = np.array(forecast_arima)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr)
                if wmape_arima < current_wmape:
                    best_model_name = f"ARIMA{order}"
                    best_error = error_arima
                    best_forecast = forecast_arima
                    current_wmape = wmape_arima
            except Exception as e:
                continue
        # Thử SARIMA nếu độ dài train đủ (> seasonal_period)
        seasonal_period = 12
        if len(train) > seasonal_period:
            for non_seasonal_order, seasonal_order in seasonal_orders:
                try:
                    model_sarima = SARIMAX(train, order=non_seasonal_order, seasonal_order=seasonal_order).fit(disp=False)
                    forecast_sarima = model_sarima.forecast(forecast_periods)
                    forecast_sarima = np.array(forecast_sarima)
                    error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                    wmape_sarima = error_sarima / np.sum(test_arr)
                    if wmape_sarima < current_wmape:
                        best_model_name = f"SARIMA{non_seasonal_order, seasonal_order}"
                        best_error = error_sarima
                        best_forecast = forecast_sarima
                        current_wmape = wmape_sarima
                except Exception as e:
                    continue

    if np.sum(test_arr) == 0:
        series_results.append({
            "Series": col,
            "Model": best_model_name,
            "wMAPE": None,
            "Status": "Invalid: Test sum zero"
        })
        continue

    status = "Valid" if current_wmape < wmape_threshold else "Invalid: wMAPE too high"

    if status == "Valid":
        overall_total_abs_error += best_error
        overall_total_actual += np.sum(test_arr)

    series_results.append({
        "Series": col,
        "Model": best_model_name,
        "wMAPE": current_wmape,
        "Status": status
    })

overall_wmape = overall_total_abs_error / overall_total_actual if overall_total_actual != 0 else np.nan

df_results = pd.DataFrame(series_results)
df_results["wMAPE (%)"] = df_results["wMAPE"].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else None)

summary = {
    "Group": "X (CV < 0.34)",
    "Total Series": len(group_x_cols),
    "Valid Series": df_results[df_results["Status"]=="Valid"].shape[0],
    "Invalid Series": df_results[df_results["Status"]!="Valid"].shape[0],
    "Overall wMAPE": f"{overall_wmape:.2%}" if pd.notnull(overall_wmape) else "NaN"
}
df_summary = pd.DataFrame([summary])

print("=== Group X - Individual Series Results (With Extended ARIMA/SARIMA Grid Search) ===")
print(df_results[["Series", "Model", "wMAPE (%)", "Status"]])
print("\n=== Group X - Summary (Valid series: wMAPE < 25%) ===")
df_summary

### Fine-tuning 1st time

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pandas.tseries.holiday import USFederalHolidayCalendar
import warnings

warnings.filterwarnings("ignore")

df_final = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df_final.index = pd.to_datetime(df_final.index)



if df_final is not None:
    # Lọc các cột thuộc nhóm z (hậu tố _z)
    group_y_cols = [col for col in df_final.columns if col != 'Total' and col.endswith('_z')]
    forecast_periods = 6
    wmape_threshold = 35  # Mục tiêu: cải thiện series có WMAPE < 35%

    # **Thêm holiday feature**
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df_final.index.min(), end=df_final.index.max())
    df_final["Holiday"] = df_final.index.isin(holidays).astype(int)

    fine_tuned_results = []

    print(f"🔄 Bắt đầu Fine-Tuning {len(group_y_cols)} series nhóm Y với holiday feature...")

    for col in group_y_cols:
        series = df_final[col].dropna()
        if len(series) < forecast_periods + 1:
            continue

        train = series.iloc[:-forecast_periods]
        test = series.iloc[-forecast_periods:]
        test_arr = np.array(test)

        train_holiday = df_final["Holiday"].iloc[:-forecast_periods]
        test_holiday = df_final["Holiday"].iloc[-forecast_periods:]

        # 🔹 Fine-tune Holt-Winters với nhiều thông số
        best_wmape = np.inf
        best_model = None
        best_forecast = None
        best_params = None

        hw_configs = [
            ("add", "add"), ("add", "mul"), ("mul", "add"), ("mul", "mul")
        ]

        for trend, seasonal in hw_configs:
            try:
                model_hw = ExponentialSmoothing(
                    train, trend=trend, seasonal=seasonal,
                    seasonal_periods=12, damped_trend=True
                ).fit()
                forecast_hw = model_hw.forecast(forecast_periods)
                error_hw = np.sum(np.abs(forecast_hw - test_arr))
                wmape_hw = error_hw / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

                if wmape_hw < best_wmape:
                    best_wmape = wmape_hw
                    best_model = model_hw
                    best_forecast = forecast_hw
                    best_params = f"Holt-Winters({trend}, {seasonal})"
            except:
                continue

        # 🔹 Fine-tune ARIMA nếu Holt-Winters chưa tốt
        if best_wmape >= wmape_threshold:
            arima_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1)]
            for order in arima_orders:
                try:
                    model_arima = ARIMA(train, order=order).fit()
                    forecast_arima = model_arima.forecast(forecast_periods)
                    error_arima = np.sum(np.abs(forecast_arima - test_arr))
                    wmape_arima = error_arima / np.sum(test_arr) * 100

                    if wmape_arima < best_wmape:
                        best_wmape = wmape_arima
                        best_model = model_arima
                        best_forecast = forecast_arima
                        best_params = f"ARIMA{order}"
                except:
                    continue

        # 🔹 Fine-tune SARIMA nếu ARIMA chưa tốt
        if best_wmape >= wmape_threshold:
            sarima_orders = [((1,1,0), (1,0,1,12)), ((0,1,1), (1,0,1,12))]
            for non_seasonal, seasonal in sarima_orders:
                try:
                    model_sarima = SARIMAX(train, order=non_seasonal, seasonal_order=seasonal).fit(disp=False)
                    forecast_sarima = model_sarima.forecast(forecast_periods)
                    error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                    wmape_sarima = error_sarima / np.sum(test_arr) * 100

                    if wmape_sarima < best_wmape:
                        best_wmape = wmape_sarima
                        best_model = model_sarima
                        best_forecast = forecast_sarima
                        best_params = f"SARIMA{non_seasonal, seasonal}"
                except:
                    continue

        # 🔹 Lưu kết quả fine-tuning
        fine_tuned_results.append({
            "Series": col,
            "Best Model": best_params,
            "New WMAPE": best_wmape,
            "Status": "Improved" if best_wmape < wmape_threshold else "Not Improved"
        })

    # **Hiển thị kết quả fine-tuning**
    df_fine_tuned = pd.DataFrame(fine_tuned_results)
    print("\n=== Fine-Tuning Results (All Y-Series with Holiday Feature & Optimized Models) ===")
    print(df_fine_tuned.to_string(index=False))

    # Hiển thị số lượng series cải thiện được
    num_improved = df_fine_tuned[df_fine_tuned["Status"] == "Improved"].shape[0]
    print(f"\n✅ Số series được cải thiện sau fine-tuning: {num_improved}/{len(group_y_cols)}")

### Fine-tuning 2nd time

In [ ]:
# 🔄 Fine-tuning lại các series chưa cải thiện bằng phương pháp nâng cao
print("\n🔄 Bắt đầu Fine-Tuning lại các series chưa đạt WMAPE < 35%...")

# **Lọc ra danh sách các series chưa đạt yêu cầu**
df_not_improved = df_fine_tuned[df_fine_tuned["Status"] == "Not Improved"]
series_to_finetune = df_not_improved["Series"].tolist()

# **Lưu kết quả fine-tuning nâng cao**
fine_tuned_results_v2 = []

for col in series_to_finetune:
    series = df_final[col].dropna()
    if len(series) < forecast_periods + 1:
        continue

    train = series.iloc[:-forecast_periods]
    test = series.iloc[-forecast_periods:]
    test_arr = np.array(test)

    train_holiday = df_final["Holiday"].iloc[:-forecast_periods]
    test_holiday = df_final["Holiday"].iloc[-forecast_periods:]

    # **🔹 Bổ sung Moving Average để làm trơn dữ liệu**
    train_smooth = train.rolling(window=3, min_periods=1).mean()

    # **🔹 Thử Holt-Winters với thông số mở rộng hơn**
    best_wmape = np.inf
    best_model = None
    best_forecast = None
    best_params = None

    hw_configs = [
        ("add", "add"), ("add", "mul"), ("mul", "add"), ("mul", "mul")
    ]

    for trend, seasonal in hw_configs:
        try:
            model_hw = ExponentialSmoothing(
                train_smooth, trend=None, seasonal=None,
                seasonal_periods=12, damped_trend=True
            ).fit()
            forecast_hw = model_hw.forecast(forecast_periods)
            error_hw = np.sum(np.abs(forecast_hw - test_arr))
            wmape_hw = error_hw / np.sum(test_arr) * 100 if np.sum(test_arr) > 0 else np.inf

            if wmape_hw < best_wmape:
                best_wmape = wmape_hw
                best_model = model_hw
                best_forecast = forecast_hw
                best_params = f"Holt-Winters({trend}, {seasonal})"
        except:
            continue

    # **🔹 Thử lại ARIMA/SARIMA nếu Holt-Winters chưa tốt**
    if best_wmape >= wmape_threshold:
        arima_orders = [(2,1,2), (1,1,2), (2,1,1), (3,1,2)]
        for order in arima_orders:
            try:
                model_arima = ARIMA(train, order=order).fit()
                forecast_arima = model_arima.forecast(forecast_periods)
                error_arima = np.sum(np.abs(forecast_arima - test_arr))
                wmape_arima = error_arima / np.sum(test_arr) * 100

                if wmape_arima < best_wmape:
                    best_wmape = wmape_arima
                    best_model = model_arima
                    best_forecast = forecast_arima
                    best_params = f"ARIMA{order}"
            except:
                continue

    # **🔹 Thử SARIMA nếu ARIMA chưa tốt**
    if best_wmape >= wmape_threshold:
        sarima_orders = [((2,1,1), (1,0,1,12)), ((1,1,2), (1,0,1,12))]
        for non_seasonal, seasonal in sarima_orders:
            try:
                model_sarima = SARIMAX(train, order=non_seasonal, seasonal_order=seasonal).fit(disp=False)
                forecast_sarima = model_sarima.forecast(forecast_periods)
                error_sarima = np.sum(np.abs(forecast_sarima - test_arr))
                wmape_sarima = error_sarima / np.sum(test_arr) * 100

                if wmape_sarima < best_wmape:
                    best_wmape = wmape_sarima
                    best_model = model_sarima
                    best_forecast = forecast_sarima
                    best_params = f"SARIMA{non_seasonal, seasonal}"
            except:
                continue

    # **🔹 Lưu kết quả fine-tuning nâng cao**
    fine_tuned_results_v2.append({
        "Series": col,
        "Best Model": best_params,
        "New WMAPE": best_wmape,
        "Status": "Improved" if best_wmape < wmape_threshold else "Not Improved"
    })

# **Hiển thị kết quả fine-tuning nâng cao**
df_fine_tuned_v2 = pd.DataFrame(fine_tuned_results_v2)
print("\n=== 🔥 Fine-Tuning Lần 2: Cải Thiện Các Series Chưa Đạt ===")
print(df_fine_tuned_v2.to_string(index=False))

# **Tính số lượng series được cải thiện sau fine-tuning lần 2**
num_improved_v2 = df_fine_tuned_v2[df_fine_tuned_v2["Status"] == "Improved"].shape[0]
print(f"\n✅ Số series được cải thiện sau Fine-Tuning lần 2: {num_improved_v2}/{len(series_to_finetune)}")

# 7.Deploy model on Gradient

In [ ]:
!pip install gradio
!pip uninstall pmdarima
!pip install --no-cache-dir pmdarima
!pip install numpy==1.25
!pip install --upgrade gradio
!python app.py

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# Modeling
from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Deep learning
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

# Gradio
import gradio as gr

#########################################
# 1. ĐỌC & CHUẨN BỊ DỮ LIỆU
#########################################
df = pd.read_csv('Series_Hierarchical.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)

group_x_cols = [col for col in df.columns if col.endswith('_x')]
group_y_cols = [col for col in df.columns if col.endswith('_y')]
group_z_cols = [col for col in df.columns if col.endswith('_z')]

# Thêm "All" vào dropdown
group_x_options = ["All"] + group_x_cols
group_y_options = ["All"] + group_y_cols
group_z_options = ["All"] + group_z_cols

#########################################
# 2. HÀM TIỆN ÍCH CHUNG
#########################################
def build_future_dates(last_date, periods):
    """Tạo dãy datetime hàng tháng, bắt đầu từ tháng tiếp theo của last_date."""
    return pd.date_range(start=last_date + pd.DateOffset(months=1),
                         periods=periods, freq="M")

def plot_history_and_forecast(history_series, forecast_df, title):
    """Trả về plotly figure hiển thị dữ liệu lịch sử (màu cam) + forecast (màu đen)."""
    fig = go.Figure()
    # Dữ liệu lịch sử (màu cam)
    fig.add_trace(go.Scatter(
        x=history_series.index,
        y=history_series.values,
        mode='lines',
        name="Historical",
        line=dict(color="orange")
    ))
    # Forecast (màu đen, nét đứt)
    fig.add_trace(go.Scatter(
        x=forecast_df["Date"],
        y=forecast_df["Forecast"],
        mode='lines',
        name="Forecast",
        line=dict(dash="dash", color="black")
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="Quantity",
        hovermode="x unified",
        margin=dict(l=40, r=20, t=60, b=40)
    )
    return fig

#########################################
# 3. PIPELINE CHO GROUP X (ĐƠN GIẢN)
#########################################
def pipeline_x_forecast(series, months=6):
    """Dự báo future cho 1 series nhóm X."""
    if len(series) < 3:
        return None

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # Ensemble (50-50)
    ensemble_pred = 0.5 * forecast_arima + 0.5 * forecast_hw

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    df_forecast = pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})
    return df_forecast

def forecast_future_x(col, months):
    """Xử lý logic forecast cho 1 cột hoặc 'All' (cộng dồn) của Group X."""
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_x_cols:
            s = df[c].dropna()
            fc = pipeline_x_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        fc = pipeline_x_forecast(s, months)
        return fc

def plot_forecast_x(col, months):
    forecast_df = forecast_future_x(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()

    if col == "All":
        # Lịch sử cộng dồn
        hist = df[group_x_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group X - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group X - {col}")
        return fig

def gradio_predict_x(col, months):
    months = int(months)
    fc_df = forecast_future_x(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_x(col, months)
    return fc_df, fig

#########################################
# 4. PIPELINE CHO GROUP Y (TƯƠNG TỰ)
#########################################
def pipeline_y_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Holt-Winters
    try:
        model_hw = ExponentialSmoothing(series, trend='add', seasonal='add', seasonal_periods=12).fit()
        forecast_hw = model_hw.forecast(months)
    except:
        forecast_hw = np.full(months, np.nan)

    # ARIMA
    try:
        model_arima = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_arima = model_arima.predict(n_periods=months)
    except:
        forecast_arima = np.full(months, np.nan)

    # XGB
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_xgb = XGBRegressor(n_estimators=50, learning_rate=0.1)
        model_xgb.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_xgb = model_xgb.predict(future_X)
    except:
        forecast_xgb = np.full(months, np.nan)

    # Ensemble (trung bình 3)
    arr_hw = np.nan_to_num(forecast_hw)
    arr_arima = np.nan_to_num(forecast_arima)
    arr_xgb = np.nan_to_num(forecast_xgb)
    ensemble_pred = (arr_hw + arr_arima + arr_xgb) / 3

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_y(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_y_cols:
            s = df[c].dropna()
            fc = pipeline_y_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_y_forecast(s, months)

def plot_forecast_y(col, months):
    forecast_df = forecast_future_y(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_y_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Y - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Y - {col}")
        return fig

def gradio_predict_y(col, months):
    months = int(months)
    fc_df = forecast_future_y(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_y(col, months)
    return fc_df, fig

#########################################
# 5. PIPELINE CHO GROUP Z (TƯƠNG TỰ)
#########################################
def pipeline_z_forecast(series, months=6):
    if len(series) < 3:
        return None

    # Model 1: SARIMAX
    try:
        model_sarimax = auto_arima(series, seasonal=True, m=12, stepwise=True, trace=False)
        forecast_sarimax = model_sarimax.predict(n_periods=months)
    except:
        forecast_sarimax = np.full(months, np.nan)

    # Model 2: GradientBoost
    try:
        X = np.arange(len(series)).reshape(-1, 1)
        y = series.values
        model_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05)
        model_gbr.fit(X, y)
        future_X = np.arange(len(series), len(series)+months).reshape(-1, 1)
        forecast_gbr = model_gbr.predict(future_X)
    except:
        forecast_gbr = np.full(months, np.nan)

    # Model 3: Ridge
    try:
        model_ridge = Ridge(alpha=1.0, random_state=42)
        model_ridge.fit(X, y)
        future_ridge = model_ridge.predict(future_X)
    except:
        future_ridge = np.full(months, np.nan)

    # Model 4: Naive
    naive_val = series.iloc[-1] if len(series) > 0 else 0
    forecast_naive = np.full(months, naive_val)

    # Ensemble median
    forecasts_stack = np.vstack([
        np.nan_to_num(forecast_sarimax),
        np.nan_to_num(forecast_gbr),
        np.nan_to_num(future_ridge),
        np.nan_to_num(forecast_naive)
    ])
    ensemble_pred = np.median(forecasts_stack, axis=0)

    last_date = series.index[-1]
    future_dates = build_future_dates(last_date, months)
    return pd.DataFrame({"Date": future_dates, "Forecast": ensemble_pred})

def forecast_future_z(col, months):
    if col == "All":
        sum_forecast = None
        base_future_dates = None
        for c in group_z_cols:
            s = df[c].dropna()
            fc = pipeline_z_forecast(s, months)
            if fc is not None:
                if sum_forecast is None:
                    sum_forecast = fc["Forecast"].values
                    base_future_dates = fc["Date"]
                else:
                    sum_forecast += fc["Forecast"].values
        if base_future_dates is None:
            return None
        return pd.DataFrame({"Date": base_future_dates, "Forecast": sum_forecast})
    else:
        s = df[col].dropna()
        return pipeline_z_forecast(s, months)

def plot_forecast_z(col, months):
    forecast_df = forecast_future_z(col, months)
    if forecast_df is None or forecast_df["Forecast"].isnull().all():
        return go.Figure()
    if col == "All":
        hist = df[group_z_cols].dropna().sum(axis=1)
        fig = plot_history_and_forecast(hist, forecast_df, "Group Z - All Series")
        return fig
    else:
        hist = df[col].dropna()
        fig = plot_history_and_forecast(hist, forecast_df, f"Group Z - {col}")
        return fig

def gradio_predict_z(col, months):
    months = int(months)
    fc_df = forecast_future_z(col, months)
    if fc_df is None:
        return pd.DataFrame({"Date": [], "Forecast": []}), go.Figure()
    fc_df["Forecast"] = fc_df["Forecast"].round(2)
    fig = plot_forecast_z(col, months)
    return fc_df, fig

#########################################
# 6. GRADIO DEPLOYMENT (3 Tabs: X, Y, Z)
#########################################

# CSS: Chỉ ép màu chữ là đen (và nền trắng) cho các phần tử text,
# KHÔNG động đến stroke/fill của các phần tử svg (Plotly lines).
custom_css = """
/* Ép text đen, nền trắng cho các element văn bản, input, button... */
body, .gradio-container, .gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container button,
.gradio-container label,
.gradio-container p,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container h5,
.gradio-container h6,
.gradio-container div,
.gradio-container span,
.gradio-container strong,
.gradio-container em {
  background-color: #FFFFFF !important;
  color: #000000 !important;
}

/* Placeholder input cũng màu đen */
::placeholder {
  color: #000000 !important;
}

/* Chiều cao cho plot */
.gradio-plot {
  height: 500px !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Multi-Group Series Forecasting Deployment for Accessories Products")

    with gr.Tabs():
        # --- TAB CHO GROUP X ---
        with gr.Tab("Group X"):
            gr.Markdown("**Forecast for columns ending with `_x`**")

            dropdown_x = gr.Dropdown(label="Select Column", choices=group_x_options, value="All")
            forecast_periods_x = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_x = gr.Button("Forecast")

            with gr.Row():
                out_table_x = gr.Dataframe(label="Forecast Table")
                out_plot_x = gr.Plot(label="Forecast Chart")

            btn_x.click(
                fn=gradio_predict_x,
                inputs=[dropdown_x, forecast_periods_x],
                outputs=[out_table_x, out_plot_x]
            )

        # --- TAB CHO GROUP Y ---
        with gr.Tab("Group Y"):
            gr.Markdown("**Forecast for columns ending with `_y`**")

            dropdown_y = gr.Dropdown(label="Select Column", choices=group_y_options, value="All")
            forecast_periods_y = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_y = gr.Button("Forecast")

            with gr.Row():
                out_table_y = gr.Dataframe(label="Forecast Table")
                out_plot_y = gr.Plot(label="Forecast Chart")

            btn_y.click(
                fn=gradio_predict_y,
                inputs=[dropdown_y, forecast_periods_y],
                outputs=[out_table_y, out_plot_y]
            )

        # --- TAB CHO GROUP Z ---
        with gr.Tab("Group Z"):
            gr.Markdown("**Forecast for columns ending with `_z`**")

            dropdown_z = gr.Dropdown(label="Select Column", choices=group_z_options, value="All")
            forecast_periods_z = gr.Slider(label="Forecast Periods (months)", minimum=1, maximum=36, value=6, step=1)
            btn_z = gr.Button("Forecast")

            with gr.Row():
                out_table_z = gr.Dataframe(label="Forecast Table")
                out_plot_z = gr.Plot(label="Forecast Chart")

            btn_z.click(
                fn=gradio_predict_z,
                inputs=[dropdown_z, forecast_periods_z],
                outputs=[out_table_z, out_plot_z]
            )

demo.launch(share=True)